### Artificial Intelligence 601.464 Project #6

### Before You Begin...
00. We're using a Jupyter Notebook environment (tutorial available here: https://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html),
01. Read the entire notebook before beginning your work, and
02.  Check the submission deadline on Gradescope.


### General Directions for this Assignment
00. Output format should be exactly as requested,
01. Functions should do only one thing.


### Before You Submit...
00. Re-read the general instructions provided above, and
01. Hit "Kernel"->"Restart & Run All". The first cell that is run should show [1], the second should show [2], and so on...
02. Submit your notebook (as .ipynb, not PDF) and your CNN model (Part 1) using Gradescope

### Part 1: Convolutional Neural Networks
For this assignment we will explore Convolution Neural Networks. The goal is classify a mushroom as either edible ('e') or poisonous ('p') based on an image. The dataset has been uploaded to Canvas. In case you'd like to learn more about it, here's the link to the repo: https://www.kaggle.com/datasets/marcosvolpato/edible-and-poisonous-fungi?select=edible+sporocarp. We are only using **sporocarps** data (mushrooms with caps). Use PyTorch (https://pytorch.org/) or its tutorial (https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html) to create your own CNN. Either start with a simple network and make it more complex. Your submission should clearly demonstrate understanding of how CNNs work.

<font color="red"> **It is recommended to run this section on colab or kaggle for GPU access.** </font>

<font color="chiffon">**GOAL: Get more than 60% accuracy!**</font>

In [1]:
# Implementation of CNN for mushroom classification

In [2]:
# CNN libraries
import torchvision
import torchvision.transforms as transforms # Preprocesse Images
from torchvision.io import decode_image # Decode Images

# NN models
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset, random_split # Dataset loading

import numpy as np # Arrays
import matplotlib.pyplot as plt # Graphing

# Progress bar
from tqdm import tqdm

import os

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

/home/jason-lafita/miniconda3/envs/ai_hw6/lib/python3.11/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [4]:
!unzip "/home/jason-lafita/ai_projects/project_6/mushroom_sporocarps.zip" # Google Colab dir

Archive:  /home/jason-lafita/ai_projects/project_6/mushroom_sporocarps.zip
  inflating: edible mushroom sporocarp/ce (1).jpeg  
  inflating: edible mushroom sporocarp/ce (1).jpg  
  inflating: edible mushroom sporocarp/ce (10).jpg  
  inflating: edible mushroom sporocarp/ce (100).jpg  
  inflating: edible mushroom sporocarp/ce (101).jpg  
  inflating: edible mushroom sporocarp/ce (102).jpg  
  inflating: edible mushroom sporocarp/ce (103).jpg  
  inflating: edible mushroom sporocarp/ce (104).jpg  
  inflating: edible mushroom sporocarp/ce (105).jpg  
  inflating: edible mushroom sporocarp/ce (106).jpg  
  inflating: edible mushroom sporocarp/ce (107).jpg  
  inflating: edible mushroom sporocarp/ce (108).jpg  
  inflating: edible mushroom sporocarp/ce (109).jpg  
  inflating: edible mushroom sporocarp/ce (11).jpg  
  inflating: edible mushroom sporocarp/ce (110).jpg  
  inflating: edible mushroom sporocarp/ce (111).jpg  
  inflating: edible mushroom sporocarp/ce (112).jpg  
  inflating:

In [5]:
edible_mushroom_path = "/home/jason-lafita/ai_projects/project_6/edible mushroom sporocarp"
poisonous_mushroom_path = "/home/jason-lafita/ai_projects/project_6/poisonous mushroom sporocarp"

In [6]:
edible_files = os.listdir(edible_mushroom_path)
poisonous_files = os.listdir(poisonous_mushroom_path)

In [7]:
# Match filename to label
edible_data = [(edible, torch.tensor(0)) for edible in edible_files]
poisonous_data= [(poison, torch.tensor(1)) for poison in poisonous_files]

In [8]:
class MushroomDataset(Dataset):
    def __init__(self, data, root_img_dir, transform=None, target_transform=None):
        self.data = data
        self.root_img_dir = root_img_dir
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_img_dir, self.data[idx][0])
        image = decode_image(img_path, "RGB")
        label = self.data[idx][1]
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)
        return image, label

In [9]:
# Standarize the size and channel values of our pictures
transform = transforms.Compose(
    [transforms.ToPILImage(),
     transforms.Resize((32, 32)),
     transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

In [10]:
edible_dataset = MushroomDataset(edible_data, edible_mushroom_path, transform=transform)
poisonous_dataset = MushroomDataset(poisonous_data, poisonous_mushroom_path, transform=transform)
dataset = ConcatDataset([edible_dataset, poisonous_dataset])

In [11]:
# Define split ratios
train_ratio = 0.7
val_ratio = 0.2
test_ratio = 0.1

# Calculate sizes for each split
train_size = int(len(dataset) * train_ratio)
val_size = int(len(dataset) * val_ratio)
test_size = len(dataset) - train_size - val_size # Ensure all data is used

# Perform the split
train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42) # For reproducibility
)

In [12]:
batch_size = 8
train_loader = DataLoader(train_dataset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size,
                                          shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size,
                                          shuffle=False, num_workers=2)

In [13]:
# Builds CNN model
class CNNet(nn.Module):
    def __init__(self):
        super().__init__()

        # Convolution layers
        self.conv1 = nn.Conv2d(
            in_channels=3, 
            out_channels=16,
            kernel_size=3,
            padding=1
        )

        self.conv2 = nn.Conv2d(
            in_channels=16,
            out_channels=32,
            kernel_size=3,
            padding=1
        )

        self.pool = nn.MaxPool2d(2,2)

        # Fully connected layers
        self.fc1 = nn.Linear(32 * 8 * 8, 128)

        # Binary classification
        self.fc2 = nn.Linear(128, 2)

    def forward(self, x):
        # Conv -> ReLU -> Pool
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))

        # Flatten
        x = torch.flatten(x, 1)

        # Fully connected layers
        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

In [14]:
# Initializes model
model = CNNet().to(device)

# Loss + optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [15]:
# Define accuracy metric
def compute_accuracy(model, data_loader):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100 * correct / total


In [16]:
# Trains model
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0

    for images, labels in tqdm(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        # Zero gradients
        optimizer.zero_grad()

        # Forward
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    

    train_acc = compute_accuracy(model, train_loader)
    val_acc = compute_accuracy(model, val_loader)

    print(f"Epoch {epoch+1}")
    print(f"Loss: {running_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.2f}%")
    print(f"Validation Accuracy: {val_acc:.2f}%")
    print("")

100%|██████████| 138/138 [00:04<00:00, 34.45it/s]


Epoch 1
Loss: 94.8192
Train Accuracy: 61.62%
Validation Accuracy: 59.37%



100%|██████████| 138/138 [00:03<00:00, 36.77it/s]


Epoch 2
Loss: 92.2805
Train Accuracy: 60.16%
Validation Accuracy: 61.27%



100%|██████████| 138/138 [00:03<00:00, 40.81it/s]


...

Epoch 8
Loss: 60.6713
Train Accuracy: 85.48%
Validation Accuracy: 66.35%



100%|██████████| 138/138 [00:04<00:00, 32.22it/s]


Epoch 9
Loss: 48.9328
Train Accuracy: 90.74%
Validation Accuracy: 64.44%



100%|██████████| 138/138 [00:04<00:00, 32.03it/s]


Epoch 10
Loss: 36.8062
Train Accuracy: 92.01%
Validation Accuracy: 67.62%



In [21]:
# Reports Final Test acc
test_acc = compute_accuracy(model, test_loader)
print(f"Final Test Accuracy: {test_acc:.2f} %")

Final Test Accuracy: 62.66 %


In [18]:
# Downloads trained model
torch.save(model.state_dict(), "mushroom_cnn.pth")
print("Model saved.")

Model saved.


Why does our CNN model perform poorly compared to our NN model (Project #5)?

CNNs can frequently perform worse than NNs when given the same amount of data. This is because a CNN usually needs a lot more data than an NN before it can start outperforming it. With this data, our smaller, dense NN from project 5 would perform better. Also, NNs are better at exploiting loopholes in data that might correspond to a higher accuracy, such as the background of the image or lighting differences, which might work really well when testing the data within the dataset, whereas the CNN needs to get a more full spatial pattern of the image. Also, CNNs can need more fine tuning that my code does not have, whereas the NN can perform well without that finetuning.

<font color="chiffon">Upload your trained model with your notebook!</font>

### Part 2: Transfer Learning
For this part we will explore transfer learning. The goal is improve our mushroom classification based on an images using a pretrained model. We will use the same dataset. **We will use ResNet34 as our base model** (https://docs.pytorch.org/vision/main/models.html), it is a popular CNN model developed by Microsoft in 2015.

Transfer learning involves using a model's pretrained weights and training only it's classification head for your task. This leverages the foundational pattern recognition of larger models while using less compute for your training.

We will use ResNet34_IMAGENET1K_V1_Weights (Default) which is a ResNet34 trained on identifying 1000 classes among 1,281,167 images.

<font color="red"> **It is recommended to run this section on colab or kaggle for GPU access.** </font>

<font color="chiffon">**GOAL: Get more than 70% accuracy!**</font>

Tips:
- Remember that transfer learning requires you to freeze your base layers!
- ResNet34 has a required input size of 224x224 pixels and a different normalization input.
- With GPU, training should take no longer than 5-10 minutes
- Consult the documentation for any issues with setup! https://docs.pytorch.org/vision/main/models/generated/torchvision.models.resnet34.html

In [19]:
from torchvision.models import resnet34, ResNet34_Weights

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Before You Submit...

00. Re-read the general instructions provided above, and
01. Submit your notebook (as .ipynb, not PDF) and your CNN model (Part 1) using Gradescope